# Tutorial LangGraph: de cero a un agente con Gemini

Este notebook es un recorrido completo (nivel basico -> intermedio) para construir grafos con LangGraph y llevarlos hasta patrones de agente reales.

La idea es aprender una pieza nueva en cada bloque y validar su resultado con salidas visibles.

Ruta del notebook (secuencial):

1. Grafo I: nodo unico (Hello World).
2. Grafo II: estado con multiples campos.
3. Grafo III: nodos en secuencia.
4. Grafo IV: bordes condicionales (routing).
5. Grafo V: bucles dentro del grafo.
6. Grafo VI: chatbot simple con Gemini.
7. Grafo VII: agente con tool calling (patron ReAct).
8. Transicion a practica avanzada (que observar antes de seguir).
9. Herramientas reales (web + SQLite + RAG local).
10. Human-in-the-loop para herramientas sensibles.
11. Memoria persistente con checkpointer (MemorySaver).
12. Multiagente con subgrafos y orquestador.

Idea clave: en LangGraph todo se modela como grafo de estados. Defines un State, nodos que transforman ese estado y edges que controlan por donde continua la ejecucion.

## 0. Preparación del entorno

Instalamos/importamos las librerías necesarias y cargamos las variables de entorno (necesitamos `GEMINI_API_KEY` para las secciones que usan Gemini).

In [ ]:
import os
import random
from typing import Annotated, Dict, List, Sequence, TypedDict

from dotenv import load_dotenv
from IPython.display import Image, display

from langgraph.graph import StateGraph, START, END

load_dotenv()  # Carga GEMINI_API_KEY (y otras variables) desde el archivo .env

## 1. Grafo I — "Hello World" (un solo nodo)

El componente más pequeño de LangGraph es un **nodo**: una función `(state) -> state` (o un diccionario con los campos que quieres actualizar del estado).

Pasos para construir cualquier grafo en LangGraph:

1. Definir el **estado** (`AgentState`), normalmente un `TypedDict`.
2. Definir uno o varios **nodos** (funciones).
3. Crear el `StateGraph`, añadir los nodos con `add_node`.
4. Definir el **punto de entrada** (`set_entry_point` o `START`) y el **punto de salida** (`set_finish_point` o `END`).
5. **Compilar** el grafo con `.compile()` para obtener una app ejecutable.
6. **Invocar** la app con `.invoke(estado_inicial)`.


In [ ]:
# El estado es la "memoria compartida" que viaja de nodo en nodo mientras se ejecuta el grafo.
class AgentState(TypedDict):
    message: str

In [ ]:
def greeting_node(state: AgentState) -> AgentState:
    """Nodo simple que añade un saludo al mensaje del estado"""
    state["message"] = "Hola " + state["message"] + ", ¿cómo va tu día?"
    return state

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("greeter", greeting_node)

graph.set_entry_point("greeter")
graph.set_finish_point("greeter")

app = graph.compile()

In [ ]:
# Visualizamos el grafo compilado (útil para depurar la topología)
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
result = app.invoke({"message": "Aldo"})
result["message"]

## 2. Grafo II — Estado con múltiples campos

El `AgentState` no tiene por qué ser un único campo: puede combinar cualquier tipo de dato (listas, strings, números...). El nodo puede leer y escribir varios campos a la vez.


In [ ]:
class AgentState(TypedDict):
    values: List[int]
    name: str
    result: str


def process_values(state: AgentState) -> AgentState:
    """Nodo que procesa varios campos del estado a la vez"""
    state["result"] = f"Hola {state['name']}! La suma de tus valores es {sum(state['values'])}"
    return state

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("processor", process_values)
graph.set_entry_point("processor")
graph.set_finish_point("processor")

app = graph.compile()
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
answers = app.invoke({"values": [1, 2, 3, 4], "name": "Aldo"})
print(answers["result"])

## 3. Grafo III — Nodos en secuencia

Ahora encadenamos **dos nodos** con `add_edge(origen, destino)`: la salida del primero alimenta al segundo. Esto es la base de cualquier *pipeline* de pasos.


In [ ]:
class AgentState(TypedDict):
    name: str
    age: str
    final: str


def first_node(state: AgentState) -> AgentState:
    """Primer nodo de la secuencia"""
    state["final"] = f"Hola {state['name']}!"
    return state


def second_node(state: AgentState) -> AgentState:
    """Segundo nodo de la secuencia: se ejecuta después del primero"""
    state["final"] += f" Tienes {state['age']} años."
    return state

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("first_node", first_node)
graph.add_node("second_node", second_node)

graph.set_entry_point("first_node")
graph.add_edge("first_node", "second_node")  # first_node → second_node
graph.set_finish_point("second_node")

app = graph.compile()
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
result = app.invoke({"name": "Charlie", "age": "20"})
print(result["final"])

## 4. Grafo IV — Bordes condicionales (*routing*)

Hasta ahora el camino entre nodos era fijo. Con `add_conditional_edges` podemos **decidir en tiempo de ejecución** a qué nodo ir, según el contenido del estado. Esto es la base del *routing* en agentes (elegir qué herramienta o rama usar).

La función de decisión (`decide_next_node`) recibe el estado y devuelve una **clave** (string); esa clave se mapea a un nodo destino mediante un diccionario.


In [ ]:
class AgentState(TypedDict):
    number1: int
    operation: str
    number2: int
    finalNumber: int


def adder(state: AgentState) -> AgentState:
    """Suma los 2 números"""
    state["finalNumber"] = state["number1"] + state["number2"]
    return state


def subtractor(state: AgentState) -> AgentState:
    """Resta los 2 números"""
    state["finalNumber"] = state["number1"] - state["number2"]
    return state


def decide_next_node(state: AgentState) -> str:
    """Decide a qué nodo ir según la operación pedida"""
    if state["operation"] == "+":
        return "addition_operation"
    elif state["operation"] == "-":
        return "subtraction_operation"

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("add_node", adder)
graph.add_node("subtract_node", subtractor)
graph.add_node("router", lambda state: state)  # nodo "de paso" que no modifica el estado

graph.add_edge(START, "router")

graph.add_conditional_edges(
    "router",
    decide_next_node,
    {
        # clave devuelta por decide_next_node: nodo destino
        "addition_operation": "add_node",
        "subtraction_operation": "subtract_node",
    },
)

graph.add_edge("add_node", END)
graph.add_edge("subtract_node", END)

app = graph.compile()
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
print(app.invoke({"number1": 10, "operation": "-", "number2": 5}))
print(app.invoke({"number1": 10, "operation": "+", "number2": 5}))

## 5. Grafo V — Bucles (*loops*)

Un borde condicional puede apuntar **al mismo nodo del que sale**: así se construye un bucle. El flujo será:

```
greeting → random → random → random → ... → END
```

La función de decisión (`should_continue`) es la que decide, en cada vuelta, si seguimos en el bucle (`"loop"`) o si salimos (`"exit"` → `END`). Es el patrón que luego usaremos para que un agente siga "pensando" hasta que ya no necesite llamar a más herramientas.


In [ ]:
class AgentState(TypedDict):
    name: str
    number: List[int]
    counter: int


def greeting_node(state: AgentState) -> AgentState:
    """Saluda a la persona e inicializa el contador"""
    state["name"] = f"Hola, {state['name']}"
    state["number"] = []
    state["counter"] = 0
    return state


def random_node(state: AgentState) -> AgentState:
    """Genera un número aleatorio entre 0 y 10 en cada vuelta del bucle"""
    state["number"].append(random.randint(0, 10))
    state["counter"] += 1
    return state


def should_continue(state: AgentState) -> str:
    """Decide si seguimos en el bucle o salimos"""
    if state["counter"] < 5:
        print("Entrando en el bucle, vuelta número:", state["counter"])
        return "loop"
    return "exit"

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("greeting", greeting_node)
graph.add_node("random", random_node)
graph.add_edge("greeting", "random")

graph.add_conditional_edges(
    "random",         # nodo origen
    should_continue,  # función de decisión
    {
        "loop": "random",  # vuelve a sí mismo → bucle
        "exit": END,
    },
)

graph.set_entry_point("greeting")

app = graph.compile()
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
app.invoke({"name": "Aldo"})

## 6. Grafo VI — Un nodo con Gemini (chatbot simple)

Hasta ahora los nodos solo manipulaban texto/números "a mano". Ahora un nodo va a **llamar a un LLM** (Gemini, vía `langchain_google_genai`) para generar la respuesta.

Para esto usamos el patrón estándar de LangGraph para conversaciones:

- El estado guarda una lista de **mensajes** (`HumanMessage`, `AIMessage`, ...).
- Usamos el *reducer* `add_messages`, que en lugar de **sobrescribir** la lista de mensajes en cada nodo, **añade** los nuevos mensajes a la lista existente.

> ⚠️ Necesitas tener definida la variable de entorno `GEMINI_API_KEY` (en un archivo `.env` en la raíz del proyecto, por ejemplo) para poder ejecutar esta sección.


In [ ]:
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph.message import add_messages

llm = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_CHAT_MODEL", "gemini-flash-latest"),
    temperature=0,
    google_api_key=os.getenv("GEMINI_API_KEY"),
)


def texto(mensaje: BaseMessage) -> str:
    """Extrae solo el texto de un mensaje (algunos modelos devuelven una lista de partes)"""
    content = mensaje.content
    if isinstance(content, str):
        return content
    return "".join(part.get("text", "") for part in content if isinstance(part, dict))

In [ ]:
class AgentState(TypedDict):
    # add_messages hace que los nuevos mensajes se acumulen en vez de reemplazar la lista
    messages: Annotated[Sequence[BaseMessage], add_messages]


def chatbot_node(state: AgentState) -> AgentState:
    """Nodo que envía el historial de mensajes a Gemini y añade su respuesta"""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("chatbot", chatbot_node)
graph.set_entry_point("chatbot")
graph.set_finish_point("chatbot")

app = graph.compile()
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
result = app.invoke({"messages": [HumanMessage(content="Explícame en una frase qué es LangGraph")]})
print(texto(result["messages"][-1]))

In [ ]:
# Gracias a add_messages, podemos seguir la conversación pasando el historial completo de vuelta
result = app.invoke({
    "messages": result["messages"] + [HumanMessage(content="¿Y en qué se diferencia de LangChain a secas?")]
})
for m in result["messages"]:
    print(f"[{m.type}] {texto(m)}\n")

## 7. Grafo VII — Agente con herramientas (tool calling + condicional + bucle)

Este es el grafo "medio": combina **todo** lo anterior en el patrón típico de un agente:

1. Definimos **herramientas** (`@tool`) que Gemini puede decidir invocar.
2. Un nodo `agent` llama a Gemini con `llm.bind_tools(tools)`. Gemini responde con texto **o** con una petición de `tool_call`.
3. Un borde condicional (`should_continue`) mira si la última respuesta tiene `tool_calls`:
   - Si **sí** → vamos al nodo `tools` (que ejecuta la herramienta) y luego **volvemos** al nodo `agent` (bucle, como en el Grafo V).
   - Si **no** → terminamos (`END`).

Este ciclo `agent → tools → agent → ... → END` es exactamente el patrón "ReAct" (razonar → actuar → observar) que usan la mayoría de agentes.


In [ ]:
from langchain_core.tools import tool


@tool
def sumar(a: float, b: float) -> float:
    """Suma dos números."""
    return a + b


@tool
def multiplicar(a: float, b: float) -> float:
    """Multiplica dos números."""
    return a * b


tools = [sumar, multiplicar]
llm_con_herramientas = llm.bind_tools(tools)

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]


def agent_node(state: AgentState) -> AgentState:
    """Le pide a Gemini el siguiente paso: responder en texto o pedir usar una herramienta"""
    response = llm_con_herramientas.invoke(state["messages"])
    return {"messages": [response]}


def should_continue(state: AgentState) -> str:
    """Si el último mensaje pide usar una herramienta, seguimos; si no, terminamos"""
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "continue"
    return "end"

In [ ]:
from langgraph.prebuilt import ToolNode

graph = StateGraph(AgentState)

graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))  # nodo ya construido por LangGraph que ejecuta las herramientas

graph.set_entry_point("agent")

graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "continue": "tools",
        "end": END,
    },
)
graph.add_edge("tools", "agent")  # tras usar la herramienta, volvemos al agente (bucle)

app = graph.compile()
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
resultado = app.invoke(
    {"messages": [HumanMessage(content="¿Cuánto es 12 más 30, y luego multiplica el resultado por 2?")]}
)

for m in resultado["messages"]:
    print(f"[{m.type}] {texto(m)}")
    if getattr(m, "tool_calls", None):
        print("   → tool_calls:", m.tool_calls)

## 8. Transicion a practica avanzada: que observar antes de seguir

Hasta aqui ya viste el patron base de un agente en LangGraph:

- `agent -> tools -> agent -> ... -> END`
- Decisiones por `tool_calls` usando bordes condicionales.
- Acumulacion de historial con `add_messages`.

Antes de pasar a la parte avanzada, fijate en estas salidas de la seccion anterior:

- En la visualizacion del grafo: aparece el bucle entre `agent` y `tools`.
- En la salida de mensajes: veras mensajes del tipo `ai`, `tool` y respuesta final del asistente.
- En `tool_calls`: el modelo decide cuando invocar herramientas y con que argumentos.

En las siguientes secciones no cambiamos esta base; la extendemos con capacidades de produccion:

1. Herramientas reales (web, DB y recuperacion tipo RAG).
2. Aprobacion humana antes de acciones sensibles.
3. Memoria entre invocaciones sin pasar manualmente todo el historial.
4. Separacion por roles con subgrafos (multiagente).

## 9. Practica avanzada A: herramientas reales (web + SQL + RAG local)

Sustituimos herramientas de juguete por herramientas cercanas a un caso real:

1. `buscar_web`: consulta informacion publica en web.
2. `consultar_db`: ejecuta consultas `SELECT` en SQLite.
3. `buscar_en_rag`: recupera contexto relevante desde un corpus local.

Que hace cada bloque de codigo de esta seccion:

- Inicializa una base SQLite en memoria y carga datos de ejemplo (`productos`).
- Declara un mini corpus para RAG local.
- Define helpers de tokenizacion/ranking para recuperar contexto.
- Expone funciones como tools de LangChain (`@tool`).

Que salida deberias ver:

- `Herramientas reales cargadas: ['buscar_web', 'consultar_db', 'buscar_en_rag']`
- Si haces una consulta SQL valida: tabla en texto con columnas y filas.
- Si no hay red en `buscar_web`: mensaje controlado de error (sin romper el flujo).

Nota tecnica: esta capa de tools es la que reutilizaremos en las siguientes secciones (HITL, memoria y multiagente).

In [ ]:
import json
import re
import sqlite3
import threading
import urllib.parse
import urllib.request
from collections import Counter

from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode

# -------------------------
# 1) Base de datos SQLite de ejemplo
# -------------------------
# check_same_thread=False evita errores cuando LangGraph ejecuta tools en otro hilo.
db_conn = sqlite3.connect(":memory:", check_same_thread=False)
db_lock = threading.Lock()
with db_lock:
    db_conn.execute(
        "CREATE TABLE productos (id INTEGER PRIMARY KEY, nombre TEXT, categoria TEXT, precio REAL)"
    )
    db_conn.executemany(
        "INSERT INTO productos (nombre, categoria, precio) VALUES (?, ?, ?)",
        [
            ("Teclado mecanico", "hardware", 79.9),
            ("Mouse inalambrico", "hardware", 34.5),
            ("Curso LangGraph Basico", "educacion", 19.0),
            ("Suscripcion RAG Pro", "software", 49.0),
        ],
    )
    db_conn.commit()

# -------------------------
# 2) Mini corpus para RAG local
# -------------------------
rag_docs = [
    "LangGraph permite modelar agentes como grafos de estado con nodos y edges condicionales.",
    "Un checkpointer como MemorySaver guarda el estado por thread_id entre invocaciones.",
    "ToolNode ejecuta tool calls de manera estructurada dentro del ciclo de un agente.",
    "Human-in-the-loop agrega control humano antes de ejecutar acciones sensibles.",
    "Un patron multiagente comun separa investigacion, planificacion y redaccion en subgrafos.",
]


def _tokens(texto_: str) -> set[str]:
    return set(re.findall(r"[a-zA-Z0-9_]+", texto_.lower()))


def _rank_rag(query: str, k: int = 2) -> list[str]:
    q = _tokens(query)
    scored: list[tuple[float, str]] = []
    for doc in rag_docs:
        d = _tokens(doc)
        if not d:
            continue
        overlap = len(q & d)
        score = overlap / max(len(q), 1)
        scored.append((score, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    top = [doc for score, doc in scored[:k] if score > 0]
    return top or rag_docs[:k]


@tool
def buscar_web(query: str) -> str:
    """Busca informacion publica en web con DuckDuckGo Instant Answer API."""
    params = urllib.parse.urlencode(
        {
            "q": query,
            "format": "json",
            "no_html": 1,
            "skip_disambig": 1,
        }
    )
    url = f"https://api.duckduckgo.com/?{params}"
    try:
        with urllib.request.urlopen(url, timeout=8) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        abstract = data.get("AbstractText") or ""
        heading = data.get("Heading") or ""
        if abstract:
            return f"{heading}: {abstract}" if heading else abstract

        snippets: list[str] = []
        for item in data.get("RelatedTopics", [])[:8]:
            if isinstance(item, dict) and item.get("Text"):
                snippets.append(item["Text"])
            elif isinstance(item, dict) and isinstance(item.get("Topics"), list):
                for sub in item["Topics"]:
                    if isinstance(sub, dict) and sub.get("Text"):
                        snippets.append(sub["Text"])
        if snippets:
            return " | ".join(snippets[:3])
        return "No encontre resultados concretos en la web para esa consulta."
    except Exception as e:
        return f"No fue posible consultar la web: {e}"


@tool
def consultar_db(sql: str) -> str:
    """Ejecuta consultas SQL de solo lectura (SELECT) sobre SQLite de ejemplo."""
    if not sql.strip().lower().startswith("select"):
        return "Consulta rechazada: solo se permiten SELECT."
    try:
        with db_lock:
            cursor = db_conn.execute(sql)
            rows = cursor.fetchall()
            cols = [c[0] for c in cursor.description] if cursor.description else []
        if not rows:
            return "Consulta valida, pero sin resultados."
        lines = [" | ".join(cols)]
        for r in rows[:10]:
            lines.append(" | ".join(map(str, r)))
        return "\n".join(lines)
    except Exception as e:
        return f"Error SQL: {e}"


@tool
def buscar_en_rag(pregunta: str, k: int = 2) -> str:
    """Recupera contexto local relevante (RAG simplificado por solapamiento de tokens)."""
    top = _rank_rag(pregunta, k=k)
    return "\n".join(f"- {doc}" for doc in top)


real_tools = [buscar_web, consultar_db, buscar_en_rag]
real_tool_node = ToolNode(real_tools)

print("Herramientas reales cargadas:", [t.name for t in real_tools])

In [ ]:
# Smoke test: ejecutar la tool de SQLite desde un hilo secundario
from concurrent.futures import ThreadPoolExecutor

def _thread_sql_call() -> str:
    return consultar_db.invoke({"sql": "SELECT nombre, precio FROM productos WHERE precio > 40"})

with ThreadPoolExecutor(max_workers=1) as ex:
    threaded_sql_result = ex.submit(_thread_sql_call).result()

print(threaded_sql_result)

## 10. Practica avanzada B: human-in-the-loop antes de herramientas sensibles

Aqui añadimos control humano explicito antes de ejecutar acciones sensibles (por ejemplo, consultar base de datos).

Topologia del flujo:

`agent -> (tools | approval | END)`

Reglas de enrutado:

- Si el modelo no pide tools: termina.
- Si pide tools no sensibles: ejecuta directo.
- Si pide tools sensibles (`consultar_db`): pasa por `approval` y solicita confirmacion por consola.

Que veras en salida:

- Un prompt de aprobacion: `¿Aprobar? (s/n)`.
- Si respondes `s`: el agente ejecuta la tool y vuelve a `agent`.
- Si respondes `n`: se corta la accion sensible y el agente responde con alternativa segura.

Comportamiento importante:

- Puede pedir aprobacion mas de una vez si el modelo decide hacer varias tool calls sensibles en una misma conversacion.
- Esto no es bug del grafo: es efecto del bucle ReAct y de la estrategia del modelo para llegar a la respuesta.

In [ ]:
llm_real_tools = llm.bind_tools(real_tools)
sensitive_tools = {"consultar_db"}


class HITLState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    approved: bool


def agent_real_node(state: HITLState) -> HITLState:
    """Nodo agente: decide si responde directamente o pide tool calls."""
    response = llm_real_tools.invoke(state["messages"])
    return {"messages": [response]}


def route_after_agent(state: HITLState) -> str:
    last = state["messages"][-1]
    tool_calls = getattr(last, "tool_calls", []) or []

    if not tool_calls:
        return "end"

    if any(tc.get("name") in sensitive_tools for tc in tool_calls):
        return "approval"

    return "tools"


def human_approval_node(state: HITLState) -> HITLState:
    """Pide confirmacion humana antes de ejecutar herramientas sensibles."""
    last = state["messages"][-1]
    pending = [tc.get("name", "<sin_nombre>") for tc in getattr(last, "tool_calls", [])]

    answer = input(
        f"Se solicita ejecutar herramientas sensibles {pending}. ¿Aprobar? (s/n): "
    ).strip().lower()
    approved = answer in {"s", "si", "sí", "y", "yes"}

    if approved:
        return {"approved": True}

    return {
        "approved": False,
        "messages": [
            AIMessage(
                content=(
                    "El usuario no aprobó la ejecución de herramientas sensibles. "
                    "Ofrece una alternativa sin consultar la base de datos."
                )
            )
        ],
    }


def route_after_approval(state: HITLState) -> str:
    return "tools" if state.get("approved") else "end"


graph_hitl = StateGraph(HITLState)
graph_hitl.add_node("agent", agent_real_node)
graph_hitl.add_node("approval", human_approval_node)
graph_hitl.add_node("tools", real_tool_node)

graph_hitl.set_entry_point("agent")
graph_hitl.add_conditional_edges(
    "agent",
    route_after_agent,
    {
        "tools": "tools",
        "approval": "approval",
        "end": END,
    },
)
graph_hitl.add_conditional_edges(
    "approval",
    route_after_approval,
    {
        "tools": "tools",
        "end": END,
    },
)
graph_hitl.add_edge("tools", "agent")

app_hitl = graph_hitl.compile()
display(Image(app_hitl.get_graph().draw_mermaid_png()))

In [ ]:
# Ejecucion sin herramienta sensible (normalmente no pide aprobacion)
demo_hitl = app_hitl.invoke(
    {
        "messages": [
            HumanMessage(
                content="Usa buscar_en_rag para resumir en 2 bullets que es LangGraph y MemorySaver."
            )
        ]
    }
)
print(texto(demo_hitl["messages"][-1]))

In [ ]:
# Prueba con herramienta sensible (activara input de aprobacion):
demo_db = app_hitl.invoke({
    "messages": [HumanMessage(content="Consulta la DB y dime productos con precio > 40")]
})
print(texto(demo_db["messages"][-1]))

In [ ]:
# Diagnostico: traza de consultas SQLite ejecutadas durante una invocacion HITL
sql_trace: list[str] = []


def _sqlite_trace(stmt: str) -> None:
    s = stmt.strip()
    if not s:
        return
    # Filtramos ruido transaccional para enfocarnos en consultas utiles.
    if s.upper().startswith(("BEGIN", "COMMIT", "ROLLBACK", "PRAGMA")):
        return
    sql_trace.append(s)


pregunta = "Consulta la DB y dime productos con precio > 40"

db_conn.set_trace_callback(_sqlite_trace)
try:
    debug_run = app_hitl.invoke({"messages": [HumanMessage(content=pregunta)]})
finally:
    db_conn.set_trace_callback(None)

print("Respuesta final del agente:\n")
print(texto(debug_run["messages"][-1]))

print("\nConsultas SQL capturadas:")
if not sql_trace:
    print("- No se capturaron consultas SQL en esta corrida.")
else:
    for i, q in enumerate(sql_trace, start=1):
        print(f"{i}. {q}")

    print("\nFrecuencia por consulta:")
    freq = Counter(sql_trace)
    for query, count in freq.items():
        print(f"- ({count}x) {query}")

## 11. Practica avanzada C: memoria persistente con checkpointer (MemorySaver)

Objetivo: mantener contexto entre invocaciones sin reenviar manualmente todo el historial.

Como funciona en esta seccion:

- Se compila el grafo con `checkpointer=MemorySaver()`.
- Se crean dos hilos logicos con `thread_id` distinto (`alumno-aldo` y `alumno-otro`).
- Se escribe un dato en un hilo y luego se pregunta por ese dato en ambos hilos.

Que deberias observar en salida:

- En `alumno-aldo`, el agente recuerda el dato guardado en el turno anterior.
- En `alumno-otro`, no recuerda ese dato (aislamiento correcto por `thread_id`).

Interpretacion:

- La memoria queda asociada al hilo, no al objeto `HumanMessage` que envies en cada llamada.
- Este patron es la base de chats multi-turno estables en produccion.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory_saver = MemorySaver()
app_hitl_memory = graph_hitl.compile(checkpointer=memory_saver)

thread_aldo = {"configurable": {"thread_id": "alumno-aldo"}}
thread_otro = {"configurable": {"thread_id": "alumno-otro"}}

# Turno 1: guardamos un dato en el hilo de Aldo
_ = app_hitl_memory.invoke(
    {
        "messages": [
            HumanMessage(
                content="Recuerda este dato para mas tarde: mi proyecto favorito es RAG clinico."
            )
        ]
    },
    config=thread_aldo,
)

# Turno 2: preguntamos en el mismo hilo (memoria persistente)
out_aldo = app_hitl_memory.invoke(
    {"messages": [HumanMessage(content="¿Cual era mi proyecto favorito?")]},
    config=thread_aldo,
)
print("[Hilo Aldo]", texto(out_aldo["messages"][-1]))

# Hilo diferente: no deberia tener ese contexto
out_otro = app_hitl_memory.invoke(
    {"messages": [HumanMessage(content="¿Cual era mi proyecto favorito?")]},
    config=thread_otro,
)
print("[Hilo Otro]", texto(out_otro["messages"][-1]))

## 12. Practica avanzada D: multiagente con subgrafos (grafo como nodo)

En esta parte componemos varios grafos para separar responsabilidades:

1. `researcher_app`: recupera contexto (RAG local).
2. `writer_app`: redacta respuesta final con ese contexto.
3. `orchestrator_app`: conecta ambos como pipeline `researcher -> writer`.

Que hace el codigo internamente:

- Cada subgrafo define su propio `State` y su nodo principal.
- El orquestador invoca subgrafos como si fueran nodos especializados.
- El estado viaja entre etapas (`question` -> `context` -> `final_answer`).

Que salida esperas ver:

- `Contexto recuperado`: bullets/documents relevantes para la pregunta.
- `Respuesta final`: texto consolidado y didactico generado por el LLM usando ese contexto.

Ventaja de arquitectura:

- Facilita pruebas, mantenimiento y escalado por rol (investigar, validar, redactar, etc.).

In [ ]:
class ResearchState(TypedDict):
    question: str
    context: str


class WriterState(TypedDict):
    question: str
    context: str
    draft: str


class OrchestratorState(TypedDict):
    question: str
    context: str
    final_answer: str


# Subgrafo 1: investigador (RAG)
def research_node(state: ResearchState) -> ResearchState:
    context = buscar_en_rag.invoke({"pregunta": state["question"], "k": 3})
    return {"context": context}


research_graph = StateGraph(ResearchState)
research_graph.add_node("research", research_node)
research_graph.set_entry_point("research")
research_graph.set_finish_point("research")
researcher_app = research_graph.compile()


# Subgrafo 2: redactor
def writer_node(state: WriterState) -> WriterState:
    prompt = (
        "Responde de forma clara y didactica.\n\n"
        f"Pregunta: {state['question']}\n\n"
        f"Contexto recuperado:\n{state['context']}\n\n"
        "Da una respuesta final breve (maximo 6 lineas)."
    )
    answer = llm.invoke([HumanMessage(content=prompt)])
    return {"draft": texto(answer)}


writer_graph = StateGraph(WriterState)
writer_graph.add_node("writer", writer_node)
writer_graph.set_entry_point("writer")
writer_graph.set_finish_point("writer")
writer_app = writer_graph.compile()


# Grafo orquestador: usa cada subgrafo como si fuera un "nodo especializado"
def run_researcher(state: OrchestratorState) -> OrchestratorState:
    out = researcher_app.invoke({"question": state["question"], "context": ""})
    return {"context": out["context"]}


def run_writer(state: OrchestratorState) -> OrchestratorState:
    out = writer_app.invoke(
        {
            "question": state["question"],
            "context": state.get("context", ""),
            "draft": "",
        }
    )
    return {"final_answer": out["draft"]}


orchestrator_graph = StateGraph(OrchestratorState)
orchestrator_graph.add_node("researcher", run_researcher)
orchestrator_graph.add_node("writer", run_writer)
orchestrator_graph.set_entry_point("researcher")
orchestrator_graph.add_edge("researcher", "writer")
orchestrator_graph.set_finish_point("writer")

orchestrator_app = orchestrator_graph.compile()
display(Image(orchestrator_app.get_graph().draw_mermaid_png()))

In [ ]:
demo_multi = orchestrator_app.invoke(
    {
        "question": "Explica para un alumno que diferencia hay entre LangChain y LangGraph",
        "context": "",
        "final_answer": "",
    }
)

print("Contexto recuperado:\n", demo_multi["context"])
print("\nRespuesta final:\n", demo_multi["final_answer"])